# Differential Equations — Session 38
## Section 8.3: Nonhomogeneous Linear Systems

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. state when matrix undetermined coefficients applies;
2. choose a suitable trial vector;
3. identify resonance with a homogeneous mode;
4. define a fundamental matrix;
5. derive variation of parameters for systems;
6. write the IVP solution formula;
7. distinguish complementary and forced responses;
8. verify analytical formulas numerically.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–15 min | Complementary plus particular solution |
| 15–38 min | Undetermined coefficients |
| 38–53 min | Resonance and trial modification |
| 53–70 min | Fundamental matrices |
| 70–84 min | Variation of parameters |
| 84–90 min | IVP formula and exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad_vec
from scipy.linalg import expm, eig
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)

def solve_linear_system(A, x0, t_span=(0, 10), points=1000, forcing=None):
    A = np.asarray(A, dtype=float)
    x0 = np.asarray(x0, dtype=float)
    t_eval = np.linspace(t_span[0], t_span[1], points)
    if forcing is None:
        def rhs(t, x): return A @ x
    else:
        def rhs(t, x): return A @ x + np.asarray(forcing(t), dtype=float)
    return solve_ivp(rhs, t_span, x0, t_eval=t_eval, rtol=1e-9, atol=1e-11)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Theorem 8.3-A — General solution

For

$$
\mathbf X'=A\mathbf X+\mathbf F(t),
$$

the general solution is

$$
\mathbf X=\mathbf X_c+\mathbf X_p,
$$

where $\mathbf X_c$ solves the associated homogeneous system.

### Method 8.3-B — Undetermined coefficients

For constant $A$, this method applies when the entries of $\mathbf F(t)$ are finite combinations of:

- constants,
- polynomials,
- exponentials,
- sines and cosines.

The trial vector must include all derivative-compatible terms.

### Principle 8.3-C — Resonance

If the proposed trial overlaps a homogeneous solution, multiply the trial by a sufficient power of $t$.

### Definition 8.3-D — Fundamental matrix

A nonsingular matrix $\Phi(t)$ satisfying

$$
\Phi'(t)=A(t)\Phi(t)
$$

is a fundamental matrix.

### Theorem 8.3-E — Variation of parameters

A particular solution is

$$
\mathbf X_p(t)
=
\Phi(t)
\int
\Phi(t)^{-1}\mathbf F(t)\,dt.
$$

### Theorem 8.3-F — IVP formula

For $\mathbf X(t_0)=\mathbf X_0$,

$$
\mathbf X(t)
=
\Phi(t)\Phi(t_0)^{-1}\mathbf X_0
+
\Phi(t)
\int_{t_0}^{t}
\Phi(\tau)^{-1}\mathbf F(\tau)\,d\tau.
$$

For constant $A$ with $\Phi(t)=e^{At}$,

$$
\mathbf X(t)
=
e^{A(t-t_0)}\mathbf X_0
+
\int_{t_0}^{t}
e^{A(t-\tau)}\mathbf F(\tau)\,d\tau.
$$

### Classroom Checkpoint — Variation of Constants

Write the constant-matrix IVP formula for

$$
\mathbf X'=A\mathbf X+\mathbf F(t),
\qquad
\mathbf X(0)=\mathbf X_0.
$$

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Constant forcing and equilibrium shift

Consider

$$
\mathbf X'=A\mathbf X+\mathbf b,
$$

with

$$
A=
\begin{pmatrix}
-2&1\\
-1&-2
\end{pmatrix},
\qquad
\mathbf b=
\begin{pmatrix}
3\\1
\end{pmatrix}.
$$

A constant particular solution satisfies

$$
A\mathbf X_p+\mathbf b=0.
$$

In [ ]:
A = np.array([[-2,1],[-1,-2]], dtype=float)
b = np.array([3,1], dtype=float)
xp = -np.linalg.solve(A, b)
print("constant particular solution:", xp)

def forcing(t):
    return b

for x0 in ([4,0], [-2,3], [0,-3]):
    sol = solve_linear_system(A, x0, (0, 8), 800, forcing)
    plt.plot(sol.y[0], sol.y[1])
plt.scatter([xp[0]], [xp[1]], s=100, label="forced equilibrium")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("Constant forcing shifts the equilibrium")
plt.show()

## 2. Polynomial forcing

For

$$
\mathbf X'=A\mathbf X+
\begin{pmatrix}
t\\1
\end{pmatrix},
$$

a natural trial is

$$
\mathbf X_p=\mathbf a\,t+\mathbf b.
$$

Substitution converts the problem into algebraic equations for the vector coefficients.

In [ ]:
t = sp.symbols("t", real=True)
A_sym = sp.Matrix([[-1,2],[0,-2]])
a1,a2,b1,b2 = sp.symbols("a1 a2 b1 b2")
a = sp.Matrix([a1,a2])
bvec = sp.Matrix([b1,b2])
Xp = a*t+bvec
F = sp.Matrix([t,1])
equations = list(sp.expand(sp.diff(Xp,t)-A_sym*Xp-F))
solution = sp.solve(
    [sp.Eq(sp.expand(eq).coeff(t, power), 0)
     for eq in equations for power in [0,1]],
    [a1,a2,b1,b2],
    dict=True
)
display(solution)

## 3. Sinusoidal forcing and frequency response

For a stable system

$$
\mathbf X'=A\mathbf X+\mathbf b\cos\omega t,
$$

the long-term response oscillates at the forcing frequency. Its amplitude depends on how close $\omega$ is to the system's natural frequencies.

In [ ]:
def sinusoidal_forcing_explorer(omega=1.0):
    A = np.array([[0,1],[-4,-0.3]], dtype=float)
    b = np.array([0,1], dtype=float)

    def forcing(t):
        return b*np.cos(omega*t)

    sol = solve_linear_system(A, [0,0], (0, 80), 5000, forcing)

    plt.plot(sol.t, sol.y[0], label="displacement-like state")
    plt.plot(sol.t, np.cos(omega*sol.t), alpha=0.5, label="input")
    plt.xlim(40, 80)
    plt.legend()
    plt.title(fr"Forced response with $\omega={omega:.2f}$")
    plt.show()

    mask = sol.t > 50
    amplitude = 0.5*(sol.y[0,mask].max()-sol.y[0,mask].min())
    print("estimated steady amplitude:", amplitude)

if WIDGETS_AVAILABLE:
    interact(
        sinusoidal_forcing_explorer,
        omega=FloatSlider(min=0.2, max=4, step=0.1, value=1)
    )
else:
    sinusoidal_forcing_explorer()

## 4. Resonance in a first-order system

Take

$$
A=
\begin{pmatrix}
1&0\\
0&-2
\end{pmatrix},
\qquad
\mathbf F(t)=e^t
\begin{pmatrix}
1\\0
\end{pmatrix}.
$$

The forcing is parallel to a homogeneous mode $e^t(1,0)^T$, so the particular solution contains $te^t$.

In [ ]:
t_grid = np.linspace(0, 3, 500)
resonant = t_grid*np.exp(t_grid)
nonresonant = (np.exp(t_grid)-np.exp(-2*t_grid))/3

plt.plot(t_grid, resonant, label=r"resonant $te^t$")
plt.plot(t_grid, nonresonant, label="nonresonant comparison")
plt.legend()
plt.title("Resonance introduces a polynomial factor")
plt.show()

## 5. Fundamental matrix properties

If

$$
\Phi(t)=e^{At},
$$

then

$$
\Phi'(t)=A\Phi(t),
\qquad
\Phi(0)=I,
\qquad
\det\Phi(t)\ne0.
$$

In [ ]:
A = np.array([[-1,2],[0,-3]], dtype=float)
t_values = np.linspace(0, 5, 300)
det_values = np.array([np.linalg.det(expm(A*t)) for t in t_values])

plt.semilogy(t_values, np.abs(det_values), label=r"$|\det e^{At}|$")
plt.plot(t_values, np.exp(np.trace(A)*t_values), linestyle="--",
         label=r"$e^{\mathrm{tr}(A)t}$")
plt.legend()
plt.title("Liouville determinant identity for constant A")
plt.show()

## 6. Variation of parameters as accumulated forcing

For zero initial data,

$$
\mathbf X(t)
=
\int_0^t e^{A(t-\tau)}\mathbf F(\tau)\,d\tau.
$$

Each past input is propagated forward by the transition matrix.

In [ ]:
A = np.array([[-1,2],[-2,-1]], dtype=float)

def F(t):
    return np.array([np.exp(-0.2*t), np.sin(t)])

def variation_of_constants(t):
    value, _ = quad_vec(lambda tau: expm(A*(t-tau)) @ F(tau), 0, t)
    return value

grid = np.linspace(0, 12, 250)
voc = np.array([variation_of_constants(t) for t in grid])

def rhs(t, x):
    return A @ x + F(t)

num = solve_ivp(rhs, (0, 12), [0,0], t_eval=grid, rtol=1e-9, atol=1e-11)

plt.plot(grid, voc[:,0], label="variation formula x")
plt.plot(grid, num.y[0], linestyle="--", label="solve_ivp x")
plt.plot(grid, voc[:,1], label="variation formula y")
plt.plot(grid, num.y[1], linestyle="--", label="solve_ivp y")
plt.legend()
plt.title("Variation of parameters verified numerically")
plt.show()

print("maximum error:", np.max(np.abs(voc-num.y.T)))

## 7. Initial condition and forced response

The IVP solution separates naturally into:

$$
\underbrace{e^{A(t-t_0)}\mathbf X_0}_{\text{zero-input response}}
+
\underbrace{\int_{t_0}^t e^{A(t-\tau)}\mathbf F(\tau)\,d\tau}_{\text{zero-state response}}.
$$

In [ ]:
def response_decomposition(scale=1.0):
    A = np.array([[-1,1],[-1,-1]], float)
    x0 = np.array([2,-1], float)
    grid = np.linspace(0, 10, 300)

    zero_input = np.array([expm(A*t) @ x0 for t in grid])

    def F(t):
        return scale*np.array([np.cos(2*t), 0])

    zero_state = np.array([
        quad_vec(lambda tau: expm(A*(t-tau)) @ F(tau), 0, t)[0]
        for t in grid
    ])

    total = zero_input+zero_state

    plt.plot(grid, zero_input[:,0], label="zero-input x")
    plt.plot(grid, zero_state[:,0], label="zero-state x")
    plt.plot(grid, total[:,0], linewidth=2, label="total x")
    plt.legend()
    plt.title("Response decomposition")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        response_decomposition,
        scale=FloatSlider(min=-3, max=3, step=0.25, value=1)
    )
else:
    response_decomposition()

## Classroom Checkpoint — Exit Check

For

$$
\mathbf X'=A\mathbf X+\mathbf F(t),
\qquad
\mathbf X(0)=\mathbf X_0,
$$

write the constant-matrix IVP formula.

> Pause here. Let students commit to an answer before running the next cell.